# Secuencias y libro de ?rdenes ? representaci?n s?, pol?tica no

Este notebook resume la parte de aprendizaje profundo del proyecto. En el workspace privado se entrenaron variantes secuenciales y de libro de ?rdenes; en el repo p?blico dejamos los scripts y los resultados clave, pero no los tensores ni modelos intermedios porque dependen del corpus completo.

La pregunta no era solo si una red aprende algo, sino si esa representaci?n se convierte en una pol?tica econ?mica estable bajo coste y validaci?n temporal.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

key = pd.read_csv(ROOT / 'results' / 'key_results.csv')
summary = pd.read_csv(ROOT / 'results' / 'final_candidate_summary.csv')
ledger = pd.read_csv(ROOT / 'results' / 'final_candidate_actions_anonymized.csv')

## Arquitecturas evaluadas

La escalera prob? primero secuencias tabulares, despu?s libro visible y finalmente fusi?n. La complejidad solo se acepta si mejora la pol?tica, no si mejora una m?trica aislada.

In [2]:
pd.DataFrame([
    {"bloque": "GRU tabular", "idea": "Secuencia de features causales", "lectura": "Se?al proxy positiva, pero con inestabilidad diaria", "estado": "NO_GO"},
    {"bloque": "Book-only Conv1D", "idea": "Tensor del libro visible sin features tabulares", "lectura": "El libro solo no generaliza en test", "estado": "NO_GO"},
    {"bloque": "Fusi?n book + tabular", "idea": "Combinar embedding del libro con secuencia tabular", "lectura": "Diagn?stico prometedor, no selector robusto", "estado": "NO_GO / diagn?stico"},
    {"bloque": "TCN / r?gimen", "idea": "Representar estados temporales y r?gimen", "lectura": "?til como pista de r?gimen, no como pol?tica final", "estado": "diagn?stico"},
])

,bloque,idea,lectura,estado
0,GRU tabular,Secuencia de features causales,"Se?al proxy positiva, pero con inestabilidad d...",NO_GO
1,Book-only Conv1D,Tensor del libro visible sin features tabulares,El libro solo no generaliza en test,NO_GO
2,Fusi?n book + tabular,Combinar embedding del libro con secuencia tab...,"Diagn?stico prometedor, no selector robusto",NO_GO / diagn?stico
3,TCN / r?gimen,Representar estados temporales y r?gimen,"?til como pista de r?gimen, no como pol?tica f...",diagn?stico


## Resultados publicados del bloque

Estos son los resultados filtrados al arco principal que se conservan en `results/key_results.csv`. Sirven para justificar por qu? el trabajo no termina en una red profunda.

In [3]:
deep_rows = key[key["phase"].isin(["sequence", "orderbook", "oof"])]
deep_rows[["phase", "experiment", "metric", "value", "unit", "status", "interpretation"]]

,phase,experiment,metric,value,unit,status,interpretation
5,sequence,tabular_gru,test_cost0p50_mean,0.4550,proxy_ticks,NO_GO,Signal remains but one bad test day
6,orderbook,tensor_audit,top10_complete,100,percent,GO,Orderbook tensor is model-ready
7,orderbook,book_only_conv1d,best_test_cost0p50_mean,-1.0830,proxy_ticks,NO_GO,Book alone is insufficient
9,oof,clean_h60_upstream,test_auc,0.5656,auc,PARTIAL_SIGNAL,Modest causal ranking signal
10,oof,clean_h60_upstream,test_top35_cost0p50_mean,0.1778,proxy_ticks,PARTIAL_SIGNAL,Positive under primary proxy cost
11,oof,clean_h60_upstream,test_top35_cost1p00_mean,-0.1899,proxy_ticks,NO_GO,Does not survive conservative cost
12,oof,clean_fusion_stage2,robust_configs,0,count,NO_GO,Second stage does not create stable policy


## Lectura

La conclusi?n es deliberadamente conservadora: las redes aportan representaci?n, pero no una pol?tica estable con el soporte disponible. Esto justifica volver a un especialista tabular m?s controlable para la decisi?n final.

In [4]:
pd.DataFrame([
    {"afirmaci?n": "El deep learning fue evaluado", "evidencia": "GRU, Conv1D, fusi?n y r?gimen aparecen en scripts/docs/resultados"},
    {"afirmaci?n": "No se promociona por est?tica", "evidencia": "book_only_conv1d queda en -1.083 ticks proxy @0.5 y la fusi?n no produce configs robustas"},
    {"afirmaci?n": "La se?al ?til se mantiene como hip?tesis", "evidencia": "H60 conserva se?al parcial, pero no coste conservador ni estabilidad suficiente"},
])

,afirmaci?n,evidencia
0,El deep learning fue evaluado,"GRU, Conv1D, fusi?n y r?gimen aparecen en scri..."
1,No se promociona por est?tica,book_only_conv1d queda en -1.083 ticks proxy @...
2,La se?al ?til se mantiene como hip?tesis,"H60 conserva se?al parcial, pero no coste cons..."
